In [10]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [11]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

In [12]:
documents = [
    Document(
        page_content="Revenue increased by 25% in Q3 due to enterprise adoption.",
        metadata={"source": "report.pdf", "topic": "finance"}
    ),
    Document(
        page_content="Customers prefer subscription-based pricing models.",
        metadata={"source": "notes.md", "topic": "product"}
    ),
    Document(
        page_content="AI platform launched with tiered pricing for mid-size companies.",
        metadata={"source": "product.json", "topic": "product"}
    )
]

In [13]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"   # latest recommended lightweight model
    ,openai_api_key=os.getenv("OPENAI_API_KEY")
)

In [14]:
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="context_demo"
)

In [15]:
query = "What pricing strategy should we use for enterprise customers?"

results = vector_store.similarity_search(
    query,
    k=2
)

for r in results:
    print("----")
    print(r.page_content)
    print(r.metadata)

----
AI platform launched with tiered pricing for mid-size companies.
{'topic': 'product', 'source': 'product.json'}
----
Customers prefer subscription-based pricing models.
{'topic': 'product', 'source': 'notes.md'}


In [16]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

docs = retriever.invoke(query)

for d in docs:
    print(d.page_content)

AI platform launched with tiered pricing for mid-size companies.
Customers prefer subscription-based pricing models.
Revenue increased by 25% in Q3 due to enterprise adoption.


In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0)

context = "\n\n".join([d.page_content for d in docs])

messages = [
    SystemMessage(content="You are a product strategist."),
    HumanMessage(content=f"""
Context:
{context}

Question:
{query}
""")
]

response = llm.invoke(messages)

print(response.content)

Given the context and the information provided, here are some strategic considerations for pricing your AI platform for enterprise customers:

1. **Value-Based Pricing**: Since enterprise customers have already contributed to a 25% revenue increase, it's clear they see significant value in your platform. Consider a value-based pricing strategy where prices are set based on the perceived value to the customer. This approach can maximize revenue by aligning the price with the benefits and ROI the enterprise customers receive.

2. **Tiered Pricing with Customization**: While you already have tiered pricing for mid-size companies, consider offering a more flexible, customizable tier for enterprise customers. This could include additional features, dedicated support, or integration services that are tailored to the specific needs of larger organizations.

3. **Subscription-Based Model**: Since customers prefer subscription-based pricing models, continue to offer this structure for enterpris